In [1]:
from collections import Counter
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import os 
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
from transformers import BertTokenizer

# load bert tokenizer from huggingface
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
def build_dict(input_batch):
  # store a batch of sparse embeddings
    sparse_emb = []
    # iterate through input batch
    for token_ids in input_batch:
        # convert the input_ids list to a dictionary of key to frequency values
        d = dict(Counter(token_ids))
        tokenids = list(set(token_ids))
        # remove special tokens and append sparse vectors to sparse_emb list
        # sparse_emb.append({key: d[key] for key in d if key not in [101, 102, 103, 0]})
        sparse_emb.append({"indices":tokenids, "values":[float(d[id]) for id in tokenids]})
    # return sparse_emb list
    return sparse_emb

In [6]:
def generate_sparse_vectors(context_batch):
    input_ids = tokenizer(
    context_batch, padding=True, truncation=True,
     max_length=512
)["input_ids"]
    sparse_embeds = build_dict(input_ids)
    return sparse_embeds

In [7]:
pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY_1000"))
index_name = "hybrid-rag"

In [8]:
indices = []
for index in pc.list_indexes():
    indices.append(index["name"])

if index_name in indices :
    print(f"index {index_name} already exists!")
    index = pc.Index(index_name)
else:
    pc.create_index(
  name=index_name,
  dimension=3072,
  metric="dotproduct",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  ),
  deletion_protection="disabled"
)
    index = pc.Index(index_name)
    print(f"index {index_name} created")



index hybrid-rag created


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=1000,
    chunk_overlap=128,
    length_function=len,
    is_separator_regex=False,
)

In [10]:
###PREPARE WEBPAGE DATA 
###Get the full text & Chunks
import json 

with open(r"../../data/url_content_mapping.json", "r",encoding="utf-8") as file:
    data = json.load(file)
texts = []
for content in data:
    texts.extend(text_splitter.create_documents([content["content"]], [{"source":content["url"]}]))


In [11]:
len(texts)

604

In [12]:
ids = [str(x) for x in range(len(texts))]
meta = [text.metadata.update({"content": text.page_content}) or text.metadata for text in texts]
content = [text.page_content for text in texts]
dense_embeds_request = client.embeddings.create(input=content,
    model="text-embedding-3-large")
dense_embeds = [item.embedding for item in dense_embeds_request.data]
sparse_embeds = generate_sparse_vectors(content)
vectors = []

for _id, sparse, dense, metadata in zip(ids, sparse_embeds, dense_embeds, meta):
        vectors.append({
            'id': _id,
            'sparse_values': sparse,
            'values': dense,
            'metadata': metadata
        })



In [13]:
vectors[0]

{'id': '0',
 'sparse_values': {'indices': [1024,
   0,
   4100,
   10760,
   3081,
   1037,
   2062,
   2063,
   20499,
   17942,
   4646,
   28712,
   2094,
   10288,
   2097,
   2102,
   2629,
   2121,
   5198,
   20562,
   101,
   102,
   2692,
   12939,
   4236,
   9871,
   4758,
   9367,
   20119,
   9883,
   27292,
   5790,
   29347,
   13995,
   18092,
   2741,
   9398,
   2239,
   7875,
   18116,
   3270,
   6855,
   10439,
   5833,
   3784,
   21197,
   5850,
   2779,
   7903,
   4319,
   4322,
   7909,
   6887,
   17641,
   4844,
   9968,
   7408,
   26354,
   12040,
   2832,
   2326,
   23319,
   5918,
   2338,
   14117,
   6442,
   3378,
   2361,
   14142,
   3401,
   2378,
   2890,
   8013,
   3413,
   2907,
   6503,
   8043,
   9587,
   2420,
   15222,
   11135,
   2433,
   8584,
   2442,
   10128,
   3477,
   2966,
   6039,
   6047,
   10665,
   2475,
   2478,
   2487,
   4037,
   17865,
   1996,
   2509,
   1998,
   1997,
   2000,
   1999,
   2003,
   2004,
   2005,
   

In [14]:
batch_size = 100

length = len(vectors) // batch_size +1
start=0
end = batch_size+1
for i in range(length):
  print(f"{start} - {end}")
  
  if end != len(vectors):
    index.upsert(vectors=vectors[start:end])
  else:
    index.upsert(vectors=vectors[start:])
  print("upserted _successfully")
  start = end 
  end = min(end +batch_size , len(vectors))


0 - 101
upserted _successfully
101 - 201
upserted _successfully
201 - 301
upserted _successfully
301 - 401
upserted _successfully
401 - 501
upserted _successfully
501 - 601
upserted _successfully
601 - 604
upserted _successfully


In [15]:
start

1278

In [16]:
1177+77

1254